# 

In [74]:
import os
import json
import shutil
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array, save_img
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

In [75]:
BASE_PATH = "/kaggle/input/cv-dataset/Project Data/Fruit"
WORK_PATH = "/kaggle/working/FruitData"

TRAIN_SRC = os.path.join(BASE_PATH, "Train")
TEST_SRC  = os.path.join(BASE_PATH, "Validation")

TRAIN_DST = os.path.join(WORK_PATH, "train")
TEST_DST  = os.path.join(WORK_PATH, "test")

os.makedirs(TRAIN_DST, exist_ok=True)
os.makedirs(TEST_DST, exist_ok=True)

In [76]:
def prepare_data(src_dir, dst_dir):
    for class_name in os.listdir(src_dir):
        class_path = os.path.join(src_dir, class_name, "Images")
        if not os.path.isdir(class_path):
            continue

        dst_class = os.path.join(dst_dir, class_name)
        os.makedirs(dst_class, exist_ok=True)

        for img in os.listdir(class_path):
            shutil.copy(os.path.join(class_path, img),
                        os.path.join(dst_class, img))

prepare_data(TRAIN_SRC, TRAIN_DST)
prepare_data(TEST_SRC, TEST_DST)

In [77]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 5e-5

TRAIN_DIR = "/kaggle/working/FruitData/train"
TEST_DIR  = "/kaggle/working/FruitData/test"

In [78]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

**Train**

In [79]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

Found 2024 images belonging to 30 classes.


**Test**

In [80]:
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 150 images belonging to 30 classes.


In [81]:
CLASS_INDICES_PATH = "/kaggle/working/class_indices.json"

with open(CLASS_INDICES_PATH, "w") as f:
    json.dump(train_generator.class_indices, f)

print("✅ class_indices.json saved at:", CLASS_INDICES_PATH)
print("Classes:", train_generator.class_indices)

✅ class_indices.json saved at: /kaggle/working/class_indices.json
Classes: {'Apple_Gala': 0, 'Apple_Golden Delicious': 1, 'Avocado': 2, 'Banana': 3, 'Berry': 4, 'Burmese Grape': 5, 'Carambola': 6, 'Date Palm': 7, 'Dragon': 8, 'Elephant Apple': 9, 'Grape': 10, 'Green Coconut': 11, 'Guava': 12, 'Hog Plum': 13, 'Kiwi': 14, 'Lichi': 15, 'Malta': 16, 'Mango Golden Queen': 17, 'Mango_Alphonso': 18, 'Mango_Amrapali': 19, 'Mango_Bari': 20, 'Mango_Himsagar': 21, 'Olive': 22, 'Orange': 23, 'Palm': 24, 'Persimmon': 25, 'Pineapple': 26, 'Pomegranate': 27, 'Watermelon': 28, 'White Pear': 29}


In [82]:
NUM_CLASSES = train_generator.num_classes
print("Classes" , train_generator.class_indices)

Classes {'Apple_Gala': 0, 'Apple_Golden Delicious': 1, 'Avocado': 2, 'Banana': 3, 'Berry': 4, 'Burmese Grape': 5, 'Carambola': 6, 'Date Palm': 7, 'Dragon': 8, 'Elephant Apple': 9, 'Grape': 10, 'Green Coconut': 11, 'Guava': 12, 'Hog Plum': 13, 'Kiwi': 14, 'Lichi': 15, 'Malta': 16, 'Mango Golden Queen': 17, 'Mango_Alphonso': 18, 'Mango_Amrapali': 19, 'Mango_Bari': 20, 'Mango_Himsagar': 21, 'Olive': 22, 'Orange': 23, 'Palm': 24, 'Persimmon': 25, 'Pineapple': 26, 'Pomegranate': 27, 'Watermelon': 28, 'White Pear': 29}


In [ ]:
base_model = EfficientNetV2B0(

    weights = "imagenet",
    include_top = False,
    input_shape = (224,224 , 3)
)
base_model.trainable = True
for layer in base_model.layers[:-30]: 
    layer.trainable = False

**output Layer**

In [84]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)

x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

In [85]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_4         │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_4     │ (None, 224, 224,  │          0 │ rescaling_4[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ normalization_4[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │      4,608 │ stem_activation[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 112, 112,  │          0 │ block1a_project_… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 56, 56,    │      9,216 │ block1a_project_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 56, 56,    │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 56, 56,    │          0 │ block2a_expand_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_co… │ (None, 56, 56,    │      2,048 │ block2a_expand_a… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_bn  │ (None, 56, 56,    │        128 │ block2a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_conv │ (None, 56, 56,    │     36,864 │ block2a_project_… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_bn   │ (None, 56, 56,    │        512 │ block2b_expand_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_act… │ (None, 56, 56,    │          0 │ block2b_expand_b

 Total params: 6,714,222 (25.61 MB)

 Trainable params: 1,958,782 (7.47 MB)

 Non-trainable params: 4,755,440 (18.14 MB)

In [86]:
WEIGHTS_PATH = "/kaggle/working/best_efficientnetv2b0.weights.h5"

callbacks = [
    EarlyStopping(
        monitor="accuracy",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="accuracy",
        factor=0.3,
        patience=3,
        min_lr=1e-7
    ),
    ModelCheckpoint(
        WEIGHTS_PATH,
        monitor="accuracy",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights = dict(enumerate(class_weights))

In [88]:
augmenter = ImageDataGenerator(
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)

class_counts = {}
total = 0

for cls in os.listdir(TRAIN_DIR):
    cls_path = os.path.join(TRAIN_DIR, cls)
    imgs = [i for i in os.listdir(cls_path)
            if i.lower().endswith(('.jpg','.png','.jpeg'))]
    class_counts[cls] = len(imgs)
    total += len(imgs)

avg = total // len(class_counts)

for cls, count in class_counts.items():
    if count >= avg:
        continue

    cls_path = os.path.join(TRAIN_DIR, cls)
    imgs = os.listdir(cls_path)
    i = 0

    while count < avg:
        img_name = np.random.choice(imgs)
        img = load_img(os.path.join(cls_path, img_name),
                       target_size=IMAGE_SIZE)
        img = img_to_array(img)
        img = np.expand_dims(img, 0)

        aug_img = next(augmenter.flow(img, batch_size=1))[0]
        save_img(os.path.join(cls_path, f"aug_{i}.jpg"), aug_img)

        count += 1
        i += 1


In [ ]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    callbacks = callbacks,
    class_weight = class_weights
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
126/127 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - accuracy: 0.0911 - loss: 3.2836
Epoch 1: accuracy improved from -inf to 0.16700, saving model to /kaggle/working/best_efficientnetv2b0.weights.h5
127/127 ━━━━━━━━━━━━━━━━━━━━ 50s 177ms/step - accuracy: 0.0923 - loss: 3.2809 - learning_rate: 5.0000e-05
Epoch 2/50
126/127 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.4679 - loss: 2.4583
Epoch 2: accuracy improved from 0.16700 to 0.52075, saving model to /kaggle/working/best_efficientnetv2b0.weights.h5
127/127 ━━━━━━━━━━━━━━━━━━━━ 9s 68ms/step - accuracy: 0.4687 - loss: 2.4551 - learning_rate: 5.0000e-05
Epoch 3/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.6863 - loss: 1.6365
Epoch 3: accuracy improved from 0.52075 to 0.72678, saving model to /kaggle/working/best_efficientnetv2b0.weights.h5
127/127 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.6867 - loss: 1.6351 - learning_rate: 5.0000e-05
Epoch 4/50
126/127 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.8117 - lo

In [90]:
test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)


10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 420ms/step - accuracy: 1.0000 - loss: 0.0011   
Test Accuracy: 1.0
Test Loss: 0.0017530304612591863
